# ORM Derivative w.r.t. Energy — ALBA-II

Tests the analytical formula for dR/dδ (δ = dp/p) derived from the
energy-scaling of lattice elements and verified via Clairaut's theorem.

**Vertical:**
$$\frac{dR^v_{ij}}{d\delta} = -\sum_{k}\frac{\partial R^v_{ij}}{\partial K_{1,k}} K_{1,k}
   - \sum_{s\in\text{sex}}\frac{\partial R^v_{ij}}{\partial K_{1,s}} K_{2,s}\,\eta_s$$

**Horizontal:**
$$\frac{dR^h_{ij}}{d\delta} = -\sum_{k}\frac{\partial R^h_{ij}}{\partial K_{1,k}} K_{1,k}
   - \sum_{k\in\text{CFD}}\frac{\partial R^h_{ij}}{\partial B_{0,k}}\cdot\frac{B_0}{K_1}\cdot K_{1,k}
   + \sum_{s\in\text{sex}}\frac{\partial R^h_{ij}}{\partial K_{1,s}} K_{2,s}\,\eta_s$$

The bending term simplifies to `dRij_dbend_thick23_disp * K1_k` because the function
already returns dR/dK0 × (B0/K1), so multiplying by K1 gives dR/dK0 × B0.

**Clairaut's theorem:** ∂²x_i/(∂θ_j ∂δ) = ∂²x_i/(∂δ ∂θ_j)  →  dR_ij/dδ = dη_i/dθ_j

Tensor convention: axis 0 = element, axis 1 = BPM, axis 2 = corrector.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import at

sys.path.insert(0, str(Path("../scripts")))

from ACalORAT import AnaORM, plot_utils, math_utils, numerical
from ACalORAT import read

## Setup

In [ ]:
lattice_path = Path("../data/ring_a2.mat")
ring, ind = read.ALBAII(lattice_path)

# Full optics for all elements (needed for sextupole dispersion)
optics_full = at.get_optics(ring, refpts=range(len(ring)))[2]

# Dispersion at sextupoles (horizontal, x-plane)
eta_sex = optics_full["dispersion"][ind["sex"], 0]  # (n_sex,)

# K2 (sextupole strength = PolynomB[2])
K2_sex  = np.array([ring[i].PolynomB[2] for i in ind["sex"]])  # (n_sex,)

# K1 for all focusing elements (quads and CFDs)
K1_quad = np.array([ring[i].PolynomB[1] for i in ind["quad"]])  # (n_quad,)
K1_CFD  = np.array([ring[i].PolynomB[1] for i in ind["CFD"]])   # (n_CFD,)

print(f"Quadrupoles : {len(ind['quad'])}")
print(f"CFDs        : {len(ind['CFD'])}")
print(f"Sextupoles  : {len(ind['sex'])}")
print(f"BPMs        : {len(ind['bpm'])}")
print(f"H correctors: {len(ind['cor']['h'])}")
print(f"V correctors: {len(ind['cor']['v'])}")
print(f"\nMax |K2·η| at sextupoles: {np.max(np.abs(K2_sex * eta_sex)):.4f}")

## Numerical dR/dδ

Semi-numerical: computes the ORM analytically at ±Δf and finite-differences.
Captures all contributions (quads, dipoles, sextupoles) automatically.

In [ ]:
dRdE = numerical.quickdORMdEnergy(ring, ind)
num_h = dRdE["h"]  # (n_bpm, n_cor_h)
num_v = dRdE["v"]  # (n_bpm, n_cor_v)
print(f"dR/dδ shape — H: {num_h.shape}, V: {num_v.shape}")

## Analytical — Vertical

Two contributions:
1. **Quads + CFDs** (K1 scaling): `−Σ_k (dR^v/dK1_k) · K1_k`
2. **Sextupoles** (chromatic kick, opposite sign to horizontal): `−Σ_s (dR^v/dK1_s) · K2_s · η_s`

In [ ]:
cORM_v = AnaORM.AnaORM(ring, "v", ind)
cORM_v.assign_optics()

# --- Quad term ---
cORM_v.quad.broadcasters(0, 3)
cORM_v.bpm.broadcasters(1, 3)
cORM_v.cor.broadcasters(2, 3)
dR_dK1_quad_v = cORM_v.dRij_dqk_thick23(cORM_v.bpm, cORM_v.cor, cORM_v.quad)  # (n_quad, n_bpm, n_cor_v)
quad_term_v = -np.einsum('kij,k->ij', dR_dK1_quad_v, K1_quad)

# --- CFD quad term (K1 component only — no vertical bending term) ---
cORM_v.CFD.broadcasters(0, 3)
dR_dK1_CFD_v = cORM_v.dRij_dqk_thick23(cORM_v.bpm, cORM_v.cor, cORM_v.CFD)   # (n_CFD, n_bpm, n_cor_v)
CFD_term_v   = -np.einsum('kij,k->ij', dR_dK1_CFD_v, K1_CFD)

# --- Sextupole term (sign is NEGATIVE in vertical) ---
cORM_v.sex.broadcasters(0, 3)
dR_dK1_sex_v = cORM_v.dRij_dqk_thick23(cORM_v.bpm, cORM_v.cor, cORM_v.sex)   # (n_sex, n_bpm, n_cor_v)
sex_term_v   = -np.einsum('kij,k->ij', dR_dK1_sex_v, K2_sex * eta_sex)

ana_v = quad_term_v + CFD_term_v + sex_term_v
print(f"Quad contribution RMS  : {np.sqrt(np.mean(quad_term_v**2)):.4e}")
print(f"CFD  contribution RMS  : {np.sqrt(np.mean(CFD_term_v**2)):.4e}")
print(f"Sext contribution RMS  : {np.sqrt(np.mean(sex_term_v**2)):.4e}")

## Results — Vertical

In [ ]:
err_v = math_utils.normalized_RMSE(num_v, ana_v)
print(f"Vertical nRMSE: {err_v:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im0 = axes[0].imshow(num_v, aspect="auto", origin="lower")
axes[0].set_title(r"Numerical $dR^v/d\delta$")
axes[0].set_xlabel("Corrector index")
axes[0].set_ylabel("BPM index")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(num_v - ana_v, aspect="auto", origin="lower")
axes[1].set_title(r"Residual (Numerical − Analytical)")
axes[1].set_xlabel("Corrector index")
axes[1].set_ylabel("BPM index")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

# Term-by-term importance
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, term, label in zip(axes2,
                            [quad_term_v, CFD_term_v, sex_term_v],
                            ["Quad (K1)", "CFD (K1)", "Sextupole (−K2η)"]):
    im = ax.imshow(term, aspect="auto", origin="lower")
    ax.set_title(label)
    ax.set_xlabel("Corrector index")
    plt.colorbar(im, ax=ax)
axes2[0].set_ylabel("BPM index")
plt.suptitle(r"Individual contributions to $dR^v/d\delta$")
plt.tight_layout()
plt.show()

## Analytical — Horizontal

Three contributions:
1. **Quads** (K1 scaling): `−Σ_k (dR^h/dK1_k) · K1_k`
2. **CFDs** (K1 + K0 scaling): `−Σ_k [(dR^h/dK1_k) + (dR^h/dB0_k)·B0/K1] · K1_k`
   — the second part is exactly `dRij_dbend_thick23_disp * K1_k`
3. **Sextupoles** (sign is POSITIVE in horizontal): `+Σ_s (dR^h/dK1_s) · K2_s · η_s`

In [ ]:
cORM_h = AnaORM.AnaORM(ring, "h", ind)
cORM_h.assign_optics()

# --- Quad term ---
cORM_h.quad.broadcasters(0, 3)
cORM_h.bpm.broadcasters(1, 3)
cORM_h.cor.broadcasters(2, 3)
dR_dK1_quad_h = cORM_h.dRij_dqk_thick23(cORM_h.bpm, cORM_h.cor, cORM_h.quad)  # (n_quad, n_bpm, n_cor_h)
quad_term_h   = -np.einsum('kij,k->ij', dR_dK1_quad_h, K1_quad)

# --- CFD term: K1 part + bending (K0) part ---
# dRij_dbend_thick23_disp returns dR/dK0 * (B0/K1), so * K1 gives dR/dK0 * B0
cORM_h.CFD.broadcasters(0, 3)
dR_dK1_CFD_h  = cORM_h.dRij_dqk_thick23(cORM_h.bpm, cORM_h.cor, cORM_h.CFD)        # (n_CFD, n_bpm, n_cor_h)
dR_dB0_CFD_h  = cORM_h.dRij_dbend_thick23_disp(cORM_h.bpm, cORM_h.cor, cORM_h.CFD) # (n_CFD, n_bpm, n_cor_h)
CFD_K1_term_h = -np.einsum('kij,k->ij', dR_dK1_CFD_h, K1_CFD)
CFD_B0_term_h = -np.einsum('kij,k->ij', dR_dB0_CFD_h, K1_CFD)  # dR/dK0 * B0 = dR_dbend * K1

# --- Sextupole term (sign is POSITIVE in horizontal) ---
cORM_h.sex.broadcasters(0, 3)
dR_dK1_sex_h = cORM_h.dRij_dqk_thick23(cORM_h.bpm, cORM_h.cor, cORM_h.sex)          # (n_sex, n_bpm, n_cor_h)
sex_term_h   = +np.einsum('kij,k->ij', dR_dK1_sex_h, K2_sex * eta_sex)

ana_h = quad_term_h + CFD_K1_term_h + CFD_B0_term_h + sex_term_h
print(f"Quad  K1 contribution RMS : {np.sqrt(np.mean(quad_term_h**2)):.4e}")
print(f"CFD   K1 contribution RMS : {np.sqrt(np.mean(CFD_K1_term_h**2)):.4e}")
print(f"CFD   B0 contribution RMS : {np.sqrt(np.mean(CFD_B0_term_h**2)):.4e}")
print(f"Sext     contribution RMS : {np.sqrt(np.mean(sex_term_h**2)):.4e}")

## Results — Horizontal

In [ ]:
err_h = math_utils.normalized_RMSE(num_h, ana_h)
print(f"Horizontal nRMSE: {err_h:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im0 = axes[0].imshow(num_h, aspect="auto", origin="lower")
axes[0].set_title(r"Numerical $dR^h/d\delta$")
axes[0].set_xlabel("Corrector index")
axes[0].set_ylabel("BPM index")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(num_h - ana_h, aspect="auto", origin="lower")
axes[1].set_title(r"Residual (Numerical − Analytical)")
axes[1].set_xlabel("Corrector index")
axes[1].set_ylabel("BPM index")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

# Term-by-term importance
fig2, axes2 = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, term, label in zip(axes2,
                            [quad_term_h, CFD_K1_term_h, CFD_B0_term_h, sex_term_h],
                            ["Quad (K1)", "CFD (K1)", "CFD (B0)", "Sextupole (+K2η)"]):
    im = ax.imshow(term, aspect="auto", origin="lower")
    ax.set_title(label)
    ax.set_xlabel("Corrector index")
    plt.colorbar(im, ax=ax)
axes2[0].set_ylabel("BPM index")
plt.suptitle(r"Individual contributions to $dR^h/d\delta$")
plt.tight_layout()
plt.show()

## Clairaut's Theorem: dR_ij/dδ = dη_i/dθ_j

Since the closed orbit satisfies x_i = R_ij θ_j and η_i = dx_i/dδ,
Clairaut's theorem gives:
$$\frac{\partial^2 x_i}{\partial \theta_j\, \partial\delta} = \frac{\partial^2 x_i}{\partial\delta\, \partial\theta_j}
\quad\Rightarrow\quad \frac{dR_{ij}}{d\delta} = \frac{d\eta_i}{d\theta_j}$$

This means the j-th column of dR/dδ tells us how the dispersion at all BPMs
changes when corrector j is kicked — without ever varying the corrector.

Verification: compute dη_i/dθ_j numerically by perturbing each corrector
and measuring the change in dispersion.

In [ ]:
# Numerical dη/dθ for horizontal correctors
# Use a small subset of correctors to keep runtime reasonable
n_cor_check = min(8, len(ind["cor"]["h"]))
cor_check   = ind["cor"]["h"][:n_cor_check]

kick_step = 1e-6  # rad
eta0_bpm  = optics_full["dispersion"][ind["bpm"], 0]  # baseline dispersion at BPMs

deta_dtheta_num = np.zeros((len(ind["bpm"]), n_cor_check))

import copy
for j, cor_idx in enumerate(cor_check):
    ring_p = copy.deepcopy(ring)
    ring_p[cor_idx].KickAngle[0] += kick_step
    optics_p = at.get_optics(ring_p, refpts=range(len(ring_p)))[2]
    eta_p = optics_p["dispersion"][ind["bpm"], 0]

    ring_m = copy.deepcopy(ring)
    ring_m[cor_idx].KickAngle[0] -= kick_step
    optics_m = at.get_optics(ring_m, refpts=range(len(ring_m)))[2]
    eta_m = optics_m["dispersion"][ind["bpm"], 0]

    deta_dtheta_num[:, j] = (eta_p - eta_m) / (2 * kick_step)

print(f"Computed dη/dθ numerically for {n_cor_check} correctors.")

In [ ]:
# Clairaut prediction: columns of dR^h/dδ should match dη/dθ
deta_dtheta_clairaut = ana_h[:, :n_cor_check]  # columns of analytical dR/dδ

err_clairaut = math_utils.normalized_RMSE(deta_dtheta_num, deta_dtheta_clairaut)
print(f"Clairaut nRMSE (dR/dδ columns vs dη/dθ numerical): {err_clairaut:.2f}%")

n_show = min(4, n_cor_check)
fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]
for ax, j in zip(axes, range(n_show)):
    ax.plot(deta_dtheta_num[:, j] * 1e3,      label=r"Numerical $d\eta/d\theta_j$")
    ax.plot(deta_dtheta_clairaut[:, j] * 1e3, "--", label=r"Clairaut: $dR_{ij}/d\delta$")
    ax.set_title(f"Corrector {j}")
    ax.set_xlabel("BPM index")
axes[0].set_ylabel(r"$d\eta_i/d\theta_j$ [mm/rad]")
axes[0].legend(fontsize=8)
plt.suptitle("Clairaut verification: dR/dδ columns = dη/dθ")
plt.tight_layout()
plt.show()

## Towards dη/dB0 via Clairaut

By the same Clairaut argument applied to bending strength:
$$\frac{d\eta_i}{dB_{0,k}} = \frac{\partial}{\partial\delta}\left(\frac{\partial x_i}{\partial B_{0,k}}\right)$$

The inner derivative ∂x_i/∂B0_k is exactly the orbit change at BPM i when
element k's bending is perturbed — which is what `dRij_dbend_thick23_disp`
captures analytically (times B0/K1 normalization).

Therefore:
$$\frac{d\eta_i}{dK_{0,k}} \approx \frac{d}{d\delta}\left[\text{Rab\_thick2\_K}(i, k) \cdot L_k\right]$$

which can be evaluated by finite-differencing the ORM-at-bending-element
formula with respect to energy — or directly from `dni_dqk_integral`
extended to bending sources (already done in the CFD dispersion notebook).

In [ ]:
# Numerical dη_i/dB0_k for a few CFDs
# B0_k = PolynomB[0]; we perturb it and measure dispersion change at BPMs.
n_CFD_check = min(4, len(ind["CFD"]))
B0_step     = 1e-6

deta_dB0_num = np.zeros((len(ind["bpm"]), n_CFD_check))

for k, cfd_idx in enumerate(ind["CFD"][:n_CFD_check]):
    ring_p = copy.deepcopy(ring)
    ring_p[cfd_idx].PolynomB[0] += B0_step
    eta_p = at.get_optics(ring_p, refpts=range(len(ring_p)))[2]["dispersion"][ind["bpm"], 0]

    ring_m = copy.deepcopy(ring)
    ring_m[cfd_idx].PolynomB[0] -= B0_step
    eta_m = at.get_optics(ring_m, refpts=range(len(ring_m)))[2]["dispersion"][ind["bpm"], 0]

    deta_dB0_num[:, k] = (eta_p - eta_m) / (2 * B0_step)

# Analytical dη/dB0: use dni_dqk_integral evaluated at BPMs
# dni_dqk_integral gives dη/dK1; for B0 perturbation we need dη/dK0 = (dη/dK1) * (K1/B0)
# Equivalently: dη/dB0 = dni_dqk_integral(bpm, CFD) * (K1_k / B0_k)
cORM_h.bpm.broadcasters(0, 2)
cORM_h.CFD.broadcasters(1, 2)
deta_dK1_h = cORM_h.dni_dqk_integral(cORM_h.bpm, cORM_h.CFD)  # (n_bpm, n_CFD)

B0_CFD = np.array([ring[i].PolynomB[0] for i in ind["CFD"]])
# dη/dK0 = (dη/dK1) / ratio = (dη/dK1) * (K1/B0)
deta_dB0_ana = deta_dK1_h[:, :n_CFD_check] * (K1_CFD[:n_CFD_check] / B0_CFD[:n_CFD_check])[None, :]

err_deta = math_utils.normalized_RMSE(deta_dB0_num, deta_dB0_ana)
print(f"dη/dB0 nRMSE (first {n_CFD_check} CFDs): {err_deta:.2f}%")

fig, axes = plt.subplots(1, n_CFD_check, figsize=(4*n_CFD_check, 4), sharey=True)
if n_CFD_check == 1:
    axes = [axes]
for ax, k in zip(axes, range(n_CFD_check)):
    ax.plot(deta_dB0_num[:, k] * 1e3,  label="Numerical")
    ax.plot(deta_dB0_ana[:, k] * 1e3, "--", label="Analytical")
    ax.set_title(f"CFD {k}")
    ax.set_xlabel("BPM index")
axes[0].set_ylabel(r"$d\eta_i/dB_{{0,k}}$ [mm / (rad/m)]")
axes[0].legend(fontsize=8)
plt.suptitle(r"Dispersion change at BPMs vs CFD bending strength")
plt.tight_layout()
plt.show()